# Simulação — Detector de Fraudes

Executa a simulação com **1.000 linhas reais** da base `fraude.xlsx`,
sem injeção de anomalias artificiais, e exibe as fraudes detectadas.

---

## Setup: caminhos e configurações

In [1]:
import sys
from pathlib import Path
from sklearn import set_config

# Garante que src/ seja encontrado a partir da pasta notebooks/
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Ativa o diagrama interativo do sklearn
set_config(display='diagram')

print(f'ROOT: {ROOT}')
print('sys.path configurado com sucesso.')

ROOT: C:\Users\Zerum IT\Documents\Zerum\EBA\fraude
sys.path configurado com sucesso.


## Carregamento e inspeção do pipeline

In [2]:
from src.predictor import FraudPredictor

predictor = FraudPredictor()
print(predictor)
print(f'\nThreshold otimizado : {predictor.threshold_:.4f}')
print(f'Tipo do modelo      : {predictor.model_type_}')

FraudPredictor(model=XGBClassifier, threshold=0.6149, status=carregado)

Threshold otimizado : 0.6149
Tipo do modelo      : XGBClassifier


In [3]:
# Diagrama interativo — clique em cada bloco para expandir os parâmetros
predictor.pipeline_

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('drop_cols', ...), ('doc2', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,cols_to_drop,"['score_fraude_modelo', 'produto', ...]"
,n_neighbors,5
,weights,'distance'


In [4]:
# Etapas do pré-processamento
preprocessing = predictor.pipeline_.named_steps['preprocessing']
print('Etapas do pipeline de pré-processamento:')
for nome, step in preprocessing.steps:
    print(f'  {nome:25s} -> {type(step).__name__}')

Etapas do pipeline de pré-processamento:
  drop_cols                 -> DropColumnsTransformer
  doc2                      -> Doc2Transformer
  pais_imputer              -> PaisImputerTransformer
  cat_grouper               -> CategoriaProdutoGrouper
  knn_imputer               -> KNNImputerNumerico
  cbe                       -> CatBoostEncoderTransformer
  kmeans_disc               -> KMeansDiscretizerTransformer
  scaler                    -> StandardScalerTransformer
  poly                      -> PolynomialFeaturesTransformer
  selector                  -> FeatureSelectorTransformer


In [5]:
# Features selecionadas pelo Boruta
selector = preprocessing.named_steps['selector']
print(f'Total de features selecionadas: {len(selector.features_present_)}')
for i, feat in enumerate(selector.features_present_, 1):
    print(f'  {i:2d}. {feat}')

Total de features selecionadas: 47
   1. pais_kmeans
   2. score_9 entrega_doc_1
   3. entrega_doc_2 valor_compra
   4. valor_compra
   5. entrega_doc_1^2
   6. score_6
   7. score_9 entrega_doc_2
   8. score_10 entrega_doc_1
   9. entrega_doc_1 entrega_doc_2
  10. entrega_doc_1_kmeans
  11. entrega_doc_2 doc_2_vazio
  12. score_1 score_2
  13. score_10_kmeans
  14. score_1 valor_compra
  15. doc_2_vazio entrega_doc_3
  16. doc_2_vazio^2
  17. score_7 score_10
  18. score_7 entrega_doc_1
  19. score_9_kmeans
  20. pais
  21. categoria_produto_kmeans
  22. score_4 score_9
  23. doc_2_vazio
  24. entrega_doc_1 entrega_doc_3
  25. doc_2_vazio_kmeans
  26. score_1
  27. score_1 score_7
  28. score_1 entrega_doc_3
  29. score_9 entrega_doc_3
  30. score_9 score_10
  31. score_6 entrega_doc_2
  32. entrega_doc_2
  33. score_9
  34. score_10
  35. entrega_doc_1
  36. score_1_kmeans
  37. categoria_produto
  38. score_1 entrega_doc_2
  39. entrega_doc_3
  40. score_7 valor_compra
  41. score_6

## Execução da simulação

In [7]:
from src.simulator import FraudSimulator

sim = FraudSimulator(n_rows=1_000, seed=42)
fraude_detected = sim.run()

In [8]:
sim.report()

  RELATÓRIO DE SIMULAÇÃO — DETECTOR DE FRAUDES
  Modelo          : XGBClassifier
  Threshold       : 0.6149
  Linhas amostradas: 1,000
  Fraudes detectadas: 136 (13.6%)
  Fraudes reais na amostra: 38
  Verdadeiros positivos  : 25

  Top 10 maiores probabilidades de fraude:
   fraude  fraude_pred  fraude_proba  valor_compra categoria_produto pais
0       1            1      0.993804          7.01       cat_0eb83e7   BR
1       1            1      0.977200         10.74       cat_e56c14f   AR
2       1            1      0.971309        121.07       cat_a287874   BR
3       1            1      0.970074         34.65       cat_5124efe   BR
4       1            1      0.960171          5.55       cat_d9753d4   BR
5       0            1      0.959691         43.28       cat_4249bd8   BR
6       0            1      0.953305         13.63       cat_ffe7351   BR
7       0            1      0.952890         37.78       cat_a132378   BR
8       1            1      0.940180         12.90       cat

In [10]:
fraude_detected.head()

,fraude,score_1,score_2,score_3,score_4,score_5,score_6,pais,score_7,produto,...,score_9,score_10,entrega_doc_1,entrega_doc_2,entrega_doc_3,data_compra,valor_compra,score_fraude_modelo,fraude_pred,fraude_proba
0,1,2,1.0000,6.85,1.0,0.000000,0.0,BR,0,Bateria Moto Moura Ma5-d - Honda Cg Fan -titan...,...,1765.0,0.0,1,N,N,2020-04-10 14:04:46,7.01,100,1,0.993804
1,1,2,0.7591,5528.84,2.0,0.000000,0.0,AR,5,Vaso Groot Para Cactos E Suculentas + Baby Groot,...,5.0,5.0,0,N,N,2020-04-04 12:57:31,10.74,88,1,0.977200
2,1,2,0.8800,54505.59,0.0,0.000000,0.0,BR,19,Kit Oi Tv Livre Digital + Antena Completa Para...,...,0.0,0.0,0,N,N,2020-03-17 15:55:44,121.07,90,1,0.971309
3,1,1,0.7895,1734026.79,1.0,0.275176,0.0,BR,29,Balcão De Cozinha Sem Pia E Tampo 2 Portas 3 G...,...,7.0,7.0,0,N,N,2020-03-08 11:23:31,34.65,83,1,0.970074
4,1,2,1.0000,5.55,1.0,0.000000,26.0,BR,0,Chaveiro Brasil Lindo,...,3548.0,1.0,1,NaN,N,2020-03-30 00:39:41,5.55,100,1,0.960171


In [11]:
# Top 10 maiores probabilidades de fraude
fraude_detected.head(10)

,fraude,score_1,score_2,score_3,score_4,score_5,score_6,pais,score_7,produto,...,score_9,score_10,entrega_doc_1,entrega_doc_2,entrega_doc_3,data_compra,valor_compra,score_fraude_modelo,fraude_pred,fraude_proba
0,1,2,1.0000,6.85,1.0,0.000000,0.0,BR,0,Bateria Moto Moura Ma5-d - Honda Cg Fan -titan...,...,1765.0,0.0,1,N,N,2020-04-10 14:04:46,7.01,100,1,0.993804
1,1,2,0.7591,5528.84,2.0,0.000000,0.0,AR,5,Vaso Groot Para Cactos E Suculentas + Baby Groot,...,5.0,5.0,0,N,N,2020-04-04 12:57:31,10.74,88,1,0.977200
2,1,2,0.8800,54505.59,0.0,0.000000,0.0,BR,19,Kit Oi Tv Livre Digital + Antena Completa Para...,...,0.0,0.0,0,N,N,2020-03-17 15:55:44,121.07,90,1,0.971309
3,1,1,0.7895,1734026.79,1.0,0.275176,0.0,BR,29,Balcão De Cozinha Sem Pia E Tampo 2 Portas 3 G...,...,7.0,7.0,0,N,N,2020-03-08 11:23:31,34.65,83,1,0.970074
4,1,2,1.0000,5.55,1.0,0.000000,26.0,BR,0,Chaveiro Brasil Lindo,...,3548.0,1.0,1,NaN,N,2020-03-30 00:39:41,5.55,100,1,0.960171
5,0,4,0.7659,16205.19,1.0,0.000000,0.0,BR,0,Tênis Sxhox Classic Deliver 4 Molas Original P...,...,0.0,0.0,0,N,N,2020-03-21 15:38:15,43.28,81,1,0.959691
6,0,4,0.7248,184550.30,1.0,0.159124,0.0,BR,7,Blusa Moletom Now United Sabina Hidalgo 99 Mus...,...,0.0,0.0,0,N,N,2020-03-30 09:11:14,13.63,80,1,0.953305
7,0,4,0.6910,63275.82,1.0,0.000000,0.0,BR,2,Kit 4 Radio Comunicador Kit 4 Walk Talk Até 12...,...,118.0,1.0,0,N,N,2020-04-10 14:23:17,37.78,83,1,0.952890
8,1,2,0.6379,834738.35,31.0,0.000000,5.0,BR,19,Free Fire 1705 Diamantes (1550 +155 Bônus) Rec...,...,1250.0,46.0,1,N,N,2020-03-10 15:41:48,12.90,93,1,0.940180
9,0,4,0.8209,3217013.43,1.0,0.000000,0.0,BR,47,Lixadeira Roto Orbital 5 Polegadas Dwe6421 Dewalt,...,2.0,2.0,0,N,N,2020-04-14 17:20:46,77.73,70,1,0.935514
